# Boolean Algebra Engine — Visualisations

Two visualisations:
1. **Complexity vs variables** — prime implicant count and eval time as variable count grows
2. **Conflict graph** — rules as nodes, edges for conflicts and equivalences

In [ ]:
!pip install git+https://github.com/Shrivastava-Aditya/boolean-algebra-engine-python.git
!pip install networkx

## 1. Complexity vs Variables

Uses XOR chains (`A^B^C^...`) — they have complex prime implicant structure and cleanly demonstrate how Quine-McCluskey scales with variable count.

In [ ]:
from core.evaluator import evaluate
from core.synthesizer import synthesize
import matplotlib.pyplot as plt

data = []
for n in range(1, 11):
    vars_ = [chr(65 + i) for i in range(n)]
    expr = "^".join(vars_)
    table, eval_m = evaluate(expr)
    _, synth_m = synthesize(table)
    data.append({
        "n": n,
        "prime_implicants": synth_m.prime_implicant_count,
        "eval_time_ms": eval_m.eval_time_ms,
        "rows": eval_m.rows_evaluated,
    })

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot([d["n"] for d in data], [d["prime_implicants"] for d in data], marker="o", color="#3fb950")
ax1.set_xlabel("Variables (n)")
ax1.set_ylabel("Prime implicant count")
ax1.set_title("QM complexity vs variables (XOR chain)")
ax1.grid(True, alpha=0.3)

ax2.plot([d["n"] for d in data], [d["eval_time_ms"] for d in data], marker="o", color="#e3b341")
ax2.set_xlabel("Variables (n)")
ax2.set_ylabel("Eval time (ms)")
ax2.set_title("Eval time vs variables (2^n rows)")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Conflict Graph

Rules as nodes. Red edges = always conflict (can never both be true). Yellow edges = equivalent (duplicate rules). No edge = compatible.

Swap in your own `rules` list to audit any rule set.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from mcp_server.server import check_prompt_logic

rules = ["A.B", "!A", "C", "A.!B", "!A+!B", "A.B+C"]

result = check_prompt_logic(rules)

G = nx.Graph()
for i, rule in enumerate(rules):
    G.add_node(i, label=rule)

for p in result["pairwise"]:
    i1 = rules.index(p["rule1"])
    i2 = rules.index(p["rule2"])
    if p["always_conflict"]:
        G.add_edge(i1, i2, kind="conflict")
    elif p["equivalent"]:
        G.add_edge(i1, i2, kind="equivalent")

pos = nx.spring_layout(G, seed=42)
conflict_edges = [(u, v) for u, v, d in G.edges(data=True) if d["kind"] == "conflict"]
equiv_edges    = [(u, v) for u, v, d in G.edges(data=True) if d["kind"] == "equivalent"]

plt.figure(figsize=(8, 6))
nx.draw_networkx_nodes(G, pos, node_color="#1f2937", node_size=2500)
nx.draw_networkx_labels(G, pos, labels={i: r for i, r in enumerate(rules)}, font_color="white", font_size=10)
nx.draw_networkx_edges(G, pos, edgelist=conflict_edges, edge_color="#f85149", width=2.5)
nx.draw_networkx_edges(G, pos, edgelist=equiv_edges,    edge_color="#e3b341", width=2.5)

plt.legend(handles=[
    plt.Line2D([0], [0], color="#f85149", linewidth=2.5, label="conflict"),
    plt.Line2D([0], [0], color="#e3b341", linewidth=2.5, label="equivalent"),
], loc="upper left")
plt.title("Rule conflict graph")
plt.axis("off")
plt.tight_layout()
plt.show()